Cell 1 — Setup

In [ ]:
import os, glob, math, random, numpy as np, cv2, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm


Cell 2 — Paths and GPU

In [ ]:
# --- set data directory
DATA_DIR = "M:/MIAMI/555/d1s1/000"   # change to folder containing 00000_radar.npz, 00000_pose.npz, etc.
OUT_DIR  = "M:/MIAMI/555/checkpoints"
os.makedirs(OUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


Cell 3 — Soft-Argmax and Model

In [ ]:
# --- Soft-Argmax
class SoftArgmax2D(nn.Module):
    def __init__(self, beta=1.0):
        super().__init__()
        self.beta = beta
    def forward(self, hms):
        B,K,H,W = hms.shape
        h = torch.softmax(hms.view(B*K, -1) * self.beta, dim=-1).view(B*K,1,H,W)
        xs = torch.linspace(0, W-1, W, device=hms.device).view(1,1,1,W)
        ys = torch.linspace(0, H-1, H, device=hms.device).view(1,1,H,1)
        x = torch.sum(h*xs, dim=[2,3]).view(B,K,1)
        y = torch.sum(h*ys, dim=[2,3]).view(B,K,1)
        coords = torch.cat([y,x], dim=-1)
        conf,_ = torch.max(h.view(B*K,-1), dim=-1)
        return coords, conf.view(B,K)

# --- simple conv backbone
def conv_bn_relu(cin, cout, k=3, s=1, p=1):
    return nn.Sequential(nn.Conv2d(cin, cout, k, s, p, bias=False),
                         nn.BatchNorm2d(cout),
                         nn.ReLU(inplace=True))

class RadarPoseNet(nn.Module):
    def __init__(self, num_kp=17, in_ch=2, base=64, out_res=128):
        super().__init__()
        self.out_res = out_res
        self.backbone = nn.Sequential(
            conv_bn_relu(in_ch, base, 7, 2, 3),
            conv_bn_relu(base, base, 3, 1, 1),
            conv_bn_relu(base, base*2, 3, 2, 1),
            conv_bn_relu(base*2, base*2, 3, 1, 1),
            conv_bn_relu(base*2, base*4, 3, 2, 1),
            conv_bn_relu(base*4, base*4, 3, 1, 1),
            conv_bn_relu(base*4, base*4, 3, 1, 1)
        )
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(base*4, base*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base*2), nn.ReLU(True),
            nn.ConvTranspose2d(base*2, base, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base), nn.ReLU(True),
            nn.ConvTranspose2d(base, base, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base), nn.ReLU(True)
        )
        self.hm_head = nn.Conv2d(base, num_kp, 1)
        self.soft = SoftArgmax2D()
    def forward(self, x):
        f = self.backbone(x)
        h = self.deconv(f)
        hm = self.hm_head(h)
        coords, conf = self.soft(hm)
        return hm, coords, conf
